In [1]:
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

print("Python executable:", sys.executable)
print("All libraries imported successfully.")

Python executable: c:\Users\anton\OneDrive\Documents\PROG8245\240926\DataColletion-PreProcessing\.venv\Scripts\python.exe
All libraries imported successfully.


### Design choices

- **Relative file path:** The dataset is located inside the repository,
  allowing the notebook to run on another computer when opened from the
  project root.
- **DataFrame:** pandas provides a tabular structure suitable for inspecting,
  cleaning, and aggregating sales records.
- **Working copy:** `raw_sales` retains the full imported dataset, while
  `sales` contains an independent copy of its first 500 rows.
- **Selection limitation:** The first 500 rows satisfy the assignment's
  permitted selection method, but they are not a random sample.
- **Initial inspection:** Row and column counts confirm the loaded dimensions.
  The three-row preview shows the structure but does not establish data quality.
- **Source preservation:** This step only reads the CSV; it does not overwrite it.

In [2]:
from pathlib import Path

# Use a relative path so the notebook also works after cloning the repository.
data_path = Path("data") / "1000 Sales Records.csv"

# Load the original file and create an independent working copy.
raw_sales = pd.read_csv(data_path)
sales = raw_sales.head(500).copy()

print(f"Rows in the source file: {len(raw_sales):,}")
print(f"Rows selected for this project: {len(sales):,}")
print(f"Number of columns: {sales.shape[1]}")

sales.head(3)

Rows in the source file: 1,000
Rows selected for this project: 500
Number of columns: 14


,Region,Country,Item Type,Sales Channel,Order Priority,Order Date,Order ID,Ship Date,Units Sold,Unit Price,Unit Cost,Total Revenue,Total Cost,Total Profit
0,Middle East and North Africa,Libya,Cosmetics,Offline,M,10/18/2014,686800706,10/31/2014,8446,437.20,263.33,3692591.20,2224085.18,1468506.02
1,North America,Canada,Vegetables,Online,M,11/7/2011,185941302,12/8/2011,3018,154.06,90.93,464953.08,274426.74,190526.34
2,Middle East and North Africa,Libya,Baby Food,Offline,C,10/31/2016,246222341,12/9/2016,1517,255.28,159.42,387259.76,241840.14,145419.62


## 2. Pick the Right Container

A dictionary suits each sales record because it provides access through
field names and allows values to be updated during cleaning, whereas a
namedtuple has fixed fields and does not support direct field reassignment.
A list can hold all sales records, while a set stores unique values,
such as the countries represented in the dataset.

In [3]:
# Convert the first row into a dictionary with column names as keys.
sample_record = sales.iloc[0].to_dict()

# Access individual values using their field names.
print("Container type:", type(sample_record).__name__)
print("Product category:", sample_record["Item Type"])
print("Unit price:", sample_record["Unit Price"])

# Display the complete record.
sample_record

Container type: dict
Product category: Cosmetics
Unit price: 437.2


{'Region': 'Middle East and North Africa',
 'Country': 'Libya',
 'Item Type': 'Cosmetics',
 'Sales Channel': 'Offline',
 'Order Priority': 'M',
 'Order Date': '10/18/2014',
 'Order ID': 686800706,
 'Ship Date': '10/31/2014',
 'Units Sold': 8446,
 'Unit Price': 437.2,
 'Unit Cost': 263.33,
 'Total Revenue': 3692591.2,
 'Total Cost': 2224085.18,
 'Total Profit': 1468506.02}

## 2. Pick the Right Container

A dictionary suits each sales record because it provides access through field names and allows values to be updated during cleaning, whereas a namedtuple has fixed fields and does not support direct field reassignment.
A list can hold all sales records, while a set stores unique values, such as the countries represented in the dataset.

In [4]:
# Convert the first row into a dictionary with column names as keys.
sample_record = sales.iloc[0].to_dict()

# Access individual values using their field names.
print("Container type:", type(sample_record).__name__)
print("Product category:", sample_record["Item Type"])
print("Unit price:", sample_record["Unit Price"])

# Display the complete record.
sample_record

Container type: dict
Product category: Cosmetics
Unit price: 437.2


{'Region': 'Middle East and North Africa',
 'Country': 'Libya',
 'Item Type': 'Cosmetics',
 'Sales Channel': 'Offline',
 'Order Priority': 'M',
 'Order Date': '10/18/2014',
 'Order ID': 686800706,
 'Ship Date': '10/31/2014',
 'Units Sold': 8446,
 'Unit Price': 437.2,
 'Unit Cost': 263.33,
 'Total Revenue': 3692591.2,
 'Total Cost': 2224085.18,
 'Total Profit': 1468506.02}

### 3. Code organization

The `SalesRecord` class and `build_sales_record()` function are implemented
in `src/sales_record.py` and imported into this notebook.
The module contains reusable logic, while the notebook documents the
workflow and displays results. The `src/__init__.py` file marks `src`
as a Python package.

In [5]:
from src.sales_record import build_sales_record

# Create a sales object using the reusable function.
first_sale = build_sales_record(sample_record)

print("Product category:", first_sale.data["Item Type"])
print(f"Calculated revenue: {first_sale.total():,.2f}")
print(f"Source revenue: {first_sale.data['Total Revenue']:,.2f}")

Product category: Cosmetics
Calculated revenue: 3,692,591.20
Source revenue: 3,692,591.20


## 4. Bulk Loaded

Convert the working DataFrame into a list of dictionaries, with one
dictionary per row. Then use `build_sales_record()` to create a
`SalesRecord` object for each dictionary.

The list preserves row order and retains repeated records so that
potential duplicates can be assessed during profiling.

In [6]:
# Convert each DataFrame row into a dictionary.
sales_dicts = sales.to_dict(orient="records")

# Build one SalesRecord object for each dictionary.
sales_records = [
    build_sales_record(record)
    for record in sales_dicts
]

# Check that the conversion preserved the number of records.
assert len(sales_records) == len(sales), "Record count mismatch."

print(f"Dictionaries created: {len(sales_dicts):,}")
print(f"SalesRecord objects created: {len(sales_records):,}")
print("First object type:", type(sales_records[0]).__name__)
print(f"First record revenue: {sales_records[0].total():,.2f}")

Dictionaries created: 500
SalesRecord objects created: 500
First object type: SalesRecord
First record revenue: 3,692,591.20
